# Arogya: train the ISL sign recogniser

This notebook trains the LSTM that recognises Indian Sign Language words from MediaPipe keypoints, using the **INCLUDE** dataset from AI4Bharat, and exports it to ONNX for the Arogya server.

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Full INCLUDE takes about 1.5 hours to extract and 10 minutes to train; INCLUDE-50 takes about 20 minutes in total.

Steps: get the code → install → download INCLUDE → extract keypoints → train → download `isl_lstm.onnx` and `labels.json`.

## 1. Get the Arogya code
The repository is private, so either paste a GitHub **fine-grained token** with read access to `AROGYA`, or upload the project zip with the file browser on the left and unzip it.

In [ ]:
import getpass, os
GITHUB_USER = "your-github-username"   # <- change this
token = getpass.getpass("GitHub token (leave empty if you uploaded the zip): ")
if token:
    !git clone https://{token}@github.com/{GITHUB_USER}/AROGYA.git
else:
    !unzip -q -o AROGYA.zip -d .
%cd AROGYA

## 2. Install training packages

In [ ]:
!pip -q install -r ml/requirements.txt

## 3. Download INCLUDE from Zenodo
INCLUDE is published at https://zenodo.org/records/4010759 (licence: see the record page). The cell lists the archive files; by default it downloads the categories that matter most in a clinic. Set `DOWNLOAD_ALL = True` for the full 263-word dataset.

In [ ]:
import requests, zipfile, pathlib, subprocess
RECORD = "4010759"
DOWNLOAD_ALL = False
WANTED = ["Greetings", "Pronouns", "Days_and_Time", "Occupations", "Adjectives", "People", "Places"]

files = requests.get(f"https://zenodo.org/api/records/{RECORD}", timeout=60).json()["files"]
for f in files:
    print(f"{f['key']:45s} {f['size']/1e9:.2f} GB")

target = pathlib.Path("/content/INCLUDE"); target.mkdir(exist_ok=True)
for f in files:
    name = f["key"]
    if not name.endswith(".zip"):
        continue
    if not DOWNLOAD_ALL and not any(name.startswith(w) for w in WANTED):
        continue
    dest = target / name
    if not dest.exists():
        print("Downloading", name)
        subprocess.run(["wget", "-q", "-O", str(dest), f["links"]["self"]], check=True)
    with zipfile.ZipFile(dest) as z:
        z.extractall(target)
    dest.unlink()
print("Done. Folders:", sorted(p.name for p in target.iterdir())[:20])

## 4. Extract keypoints
Add `--include50` to use only the 50-word benchmark subset, or `--limit 40` for a 2-minute smoke test.

In [ ]:
!python ml/extract_keypoints.py --data /content/INCLUDE --out ml/data/keypoints.npz

## 5. Train and export

In [ ]:
!python ml/train_lstm.py --data ml/data/keypoints.npz --out backend/app/data/models --epochs 80

In [ ]:
import json
r = json.load(open("backend/app/data/models/report.json"))
print(f"Accuracy {r['accuracy']:.1%}, top-3 {r['top3']:.1%} on {r['test_samples']} held-out videos, {r['classes']} classes")
print("Most confused pairs:", r["most_confused"][:8])

## 6. Download the model
Put both files in `backend/app/data/models/` of the repository and push. On the next deploy, `/api/health` will show `sign_model: true`, and the recogniser will combine the LSTM with your taught templates.

In [ ]:
from google.colab import files
files.download("backend/app/data/models/isl_lstm.onnx")
files.download("backend/app/data/models/labels.json")
files.download("backend/app/data/models/report.json")